In [2]:
import psi4
import pandas as pd
import os
import numpy as np
from lps_rscf import lps_solver

In [3]:
csv_file = 'chan2001_table1.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> Loaded existing results: 20 rows found.


In [6]:
psi4.core.set_output_file('output.dat', False)

ATOMS = {
    'H':  {'mult': 2}, 
    'He': {'mult': 1},
    'Li': {'mult': 2}, 
    'Be': {'mult': 1},
    'B':  {'mult': 2}, 
    'C':  {'mult': 3},
    'N':  {'mult': 4}, 
    'O':  {'mult': 3},
    'F':  {'mult': 2}, 
    'Ne': {'mult': 1}
}

METHOD = "TFD0.111111W"
TP = ['LDA_K_TF', 1.0]
LAMBDA = 0.111111
EXC = ['LDA_X', 1.0, 'LDA_C_VWN', 0.0]
FA = [False, 1.0]
DIIS = True
MAX_ITER = 3000
DAMPING = [0.9, 0.9, 0.001]
D_guess = None
verbose=False

psi4.set_options({'basis': 'Chan2001', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

for atom in ATOMS:
    
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    try:
        E, D, mu, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,MOL,DAMPING,FA,D_guess,DIIS,verbose)
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print(f"Calculated Energy: {E:.4f} Hartree")
            row = {
                "Method": METHOD,
                "Atom": atom,
                "Basis": psi4.core.get_global_option("BASIS"),
                "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
                "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
                "Energy,Ha": E,
                "ChemPot,Ha": mu,
                "Iterations": iterations,
                "DIIS": DIIS,
                "Damp_Start": DAMPING[0],
                "Damp_End": DAMPING[1],
                "Damp_Cutoff": DAMPING[2]
            }
            
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    
    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue

Calculating H with TFD0.111111W...
Calculated Energy: -0.6664 Hartree
Calculating He with TFD0.111111W...
Calculated Energy: -3.2228 Hartree
Calculating Li with TFD0.111111W...
Calculated Energy: -8.2515 Hartree
Calculating Be with TFD0.111111W...
Calculated Energy: -16.1631 Hartree
Calculating B with TFD0.111111W...
Calculated Energy: -27.2876 Hartree
Calculating C with TFD0.111111W...
Calculated Energy: -41.9052 Hartree
Calculating N with TFD0.111111W...
Calculated Energy: -60.2623 Hartree
Calculating O with TFD0.111111W...
Calculated Energy: -82.5798 Hartree
Calculating F with TFD0.111111W...
Calculated Energy: -109.0593 Hartree
Calculating Ne with TFD0.111111W...
Calculated Energy: -139.8865 Hartree


In [19]:
# print(df.loc[df['Method'] == 'TFD0.111111W', 'ChemPot,Ha'].map('{:.4f}'.format))
df

,Method,Atom,Basis,Grid_Sph,Grid_Rad,"Energy,Ha","ChemPot,Ha",Iterations,DIIS,Damp_Start,Damp_End,Damp_Cutoff
0,TFDW,H,CHAN2001,6,1000,-0.261824,-0.071420,14,True,0.1,0.0,0.001
1,TFDW,He,CHAN2001,6,1000,-1.477441,-0.108245,14,True,0.1,0.0,0.001
2,TFDW,Li,CHAN2001,6,1000,-4.105405,-0.130595,15,True,0.1,0.0,0.001
3,TFDW,Be,CHAN2001,6,1000,-8.492155,-0.145317,18,True,0.1,0.0,0.001
4,TFDW,B,CHAN2001,6,1000,-14.925837,-0.155647,51,True,0.1,0.0,0.001
5,TFDW,C,CHAN2001,6,1000,-23.656811,-0.163258,44,True,0.1,0.0,0.001
6,TFDW,O,CHAN2001,6,1000,-48.883117,-0.173744,133,True,0.9,0.0,0.001
7,TFDW,N,CHAN2001,6,1000,-34.908350,-0.169104,104,True,0.9,0.0,0.001
8,TFDW,F,CHAN2001,6,1000,-65.767433,-0.177532,601,True,0.1,0.0,0.001
9,TFDW,Ne,CHAN2001,6,1000,-85.734275,-0.180699,126,True,0.9,0.0,0.001


In [9]:
df.to_csv("chan2001_table1.csv", index=False)

In [ ]:
titles = list(df.index)
titles[6], titles[8] = titles[8], titles[6]
df = df.reindex(titles)
display(df)